# Tecnológico Nacional de México
## Instituto Tecnológico de Ciudad Madero
### Departamento de Sistemas y Computación | Ingeniería en Sistemas Computacionales
---
**Asignatura:** Tópicos Avanzados de Programación  
**Docente:** Alejandro Estrada Padilla  
**Práctica de laboratorio:** Análisis y visualización de calificaciones con Pandas y Matplotlib  
**Estudiante:** Emiliano Montes Morales  
**Fecha:** Octubre 2026

### 1. Objetivo
Al terminar la práctica, el estudiante será capaz de cargar un archivo CSV en un DataFrame, filtrar registros, construir columnas, resumir información con groupby y comunicar los resultados con tres gráficas de Matplotlib.

## 0. Importación de Librerías y Configuración del Entorno
Importamos las bibliotecas requeridas: `pandas` para estructuración tabular de datos, `numpy` para vectorización condicional eficiente y `matplotlib.pyplot` para la visualización estadística.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración estética general para las gráficas
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})
print('Entorno y librerías cargadas con éxito.')

## Ejercicio 1. Carga y revisión
**Lógica aplicada:**  
Se carga el conjunto de datos `calificaciones_isc.csv` en un DataFrame llamado `df`. Se revisan sus primeras filas (`head()`), sus dimensiones (`shape`), los tipos de dato de cada columna (`dtypes`) y el resumen estadístico con `describe()` para `calificacion` y `asistencias`.

> **Comentario analítico sobre el tamaño del conjunto:**  
> ¿Por qué el archivo tiene 288 registros y no 24?  
> La generación tiene 24 alumnos en total, pero el dataset registra la calificación por unidad temática y por materia. Cada alumno cursa 4 materias (`Programacion`, `Bases de datos`, `Redes` y `Matematicas discretas`) y cada una se evalúa a lo largo de 3 unidades. Por lo tanto, cada estudiante aporta $4 \times 3 = 12$ filas de registro. Multiplicando $24 \text{ alumnos} \times 12 \text{ registros} = 288$ registros en total.

In [ ]:
# Carga del archivo
df = pd.read_csv('calificaciones_isc.csv')

# Primeras 5 filas
print('--- Primeras 5 filas ---')
display(df.head())

# Dimensiones
print(f'\nDimensiones del DataFrame (filas, columnas): {df.shape}')
print(f'Total de filas: {df.shape[0]} | Total de columnas: {df.shape[1]}')

# Tipos de datos
print('\n--- Tipo de dato de cada columna ---')
print(df.dtypes)

# Resumen estadístico
print('\n--- Resumen estadístico de calificación y asistencias ---')
display(df[['calificacion', 'asistencias']].describe())

## Ejercicio 2. Filtros
**Lógica aplicada:**  
Se extraen subconjuntos mediante máscaras booleanas sin mutar el DataFrame original `df`. Para responder al conteo solicitado por la rúbrica, se emplea estrictamente la función `len(df_filtrado)`, evitando ciclos `for` iterativos.

In [ ]:
# 1. Solo el grupo A
filtro_grupo_a = df[df['grupo'] == 'A']
conteo_grupo_a = len(filtro_grupo_a)

# 2. Unidades con calificación menor a 70
filtro_menor_70 = df[df['calificacion'] < 70]
conteo_menor_70 = len(filtro_menor_70)

# 3. Bases de datos, del grupo B, con calificación >= 80
filtro_bd_b_80 = df[
    (df['materia'] == 'Bases de datos') & 
    (df['grupo'] == 'B') & 
    (df['calificacion'] >= 80)
]
conteo_bd_b_80 = len(filtro_bd_b_80)

print(f'[2.1] Filas del Grupo A: {conteo_grupo_a}')
print(f'[2.2] Filas con calificación reprobatoria (< 70): {conteo_menor_70}')
print(f'[2.3] Filas de Bases de datos en Grupo B con calif >= 80: {conteo_bd_b_80}')

## Ejercicio 3. Columna calculada
**Lógica aplicada:**  
Se añade la columna `situacion` aprovechando la operación vectorizada `np.where()`. Si la calificación es mayor o igual a 70 se categoriza como `'Aprobado'`; de lo contrario, se marca como `'Reprobado'`.

In [ ]:
# Columna calculada vectorizada
df['situacion'] = np.where(df['calificacion'] >= 70, 'Aprobado', 'Reprobado')

print('Columna situacion creada con éxito:')
display(df[['matricula', 'nombre', 'materia', 'unidad', 'calificacion', 'situacion']].head(8))

## Ejercicio 4. Resúmenes y Exportación
**Lógica aplicada:**  
Se efectúan agrupaciones con `.groupby()` para sintetizar promedios por materia, por grupo y materia, por alumno y por unidad. Para la exportación de `promedios_alumnos.csv`, se aplica `.reset_index()` para convertir los índices del grupo en columnas de datos y se pasa `index=False` a `.to_csv()` para prevenir la inclusión de columnas numéricas redundantes.

In [ ]:
# 1. Promedio por materia
promedio_materia = df.groupby('materia')['calificacion'].mean()
print('=== [4.1] Promedio por Materia ===')
display(promedio_materia.round(2))

# 2. Promedio por grupo y materia
promedio_grupo_materia = df.groupby(['grupo', 'materia'])['calificacion'].mean()
print('\n=== [4.2] Promedio por Grupo y Materia ===')
display(promedio_grupo_materia.round(2))

# 3. Promedio general de cada alumno, ordenado de mayor a menor
promedios_alumnos = (
    df.groupby(['matricula', 'nombre'])['calificacion']
    .mean()
    .reset_index()
    .rename(columns={'calificacion': 'promedio'})
    .sort_values(by='promedio', ascending=False)
)
print('\n=== [4.3] Promedio General por Alumno (Top 5) ===')
display(promedios_alumnos.head(5))

# 4. Promedio general por unidad
promedio_unidad = df.groupby('unidad')['calificacion'].mean()
print('\n=== [4.4] Promedio General por Unidad ===')
display(promedio_unidad.round(2))

# 5. Conteo de unidades aprobadas y reprobadas
conteo_situacion = df['situacion'].value_counts()
print('\n=== [4.5] Conteo Total de Aprobadas vs Reprobadas ===')
display(conteo_situacion)

# Exportación obligatoria a CSV con reset_index() e index=False
promedios_alumnos.to_csv('promedios_alumnos.csv', index=False)
print('\n[4.6] Archivo promedios_alumnos.csv exportado exitosamente.')

## Ejercicio 5. Gráficas
**Lógica y Prevención Crítica:**  
Se construyen tres visualizaciones claras incorporando rejilla (`plt.grid(True, linestyle="--", alpha=0.6)`), títulos y etiquetas en ambos ejes.  
> **Pauta crítica de rúbrica:** Se ejecuta `plt.savefig()` estrictamente antes de `plt.show()` para evitar la generación de imágenes en blanco.

In [ ]:
# --- Gráfica 1: Barras del promedio por materia ---
fig1, ax1 = plt.subplots(figsize=(8, 5))
colores = ['#2b5c8f', '#d95f02', '#7570b3', '#1b9e77']

barras = ax1.bar(
    promedio_materia.index, 
    promedio_materia.values, 
    color=colores, 
    edgecolor='black', 
    alpha=0.85,
    width=0.55
)

ax1.set_title('Promedio General de Calificación por Materia', pad=15, fontweight='bold')
ax1.set_xlabel('Materia Evaluada', labelpad=10, fontweight='bold')
ax1.set_ylabel('Promedio de Calificación (Escala 0-100)', labelpad=10, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(True, linestyle='--', alpha=0.6, axis='y')

for barra in barras:
    altura = barra.get_height()
    ax1.annotate(
        f'{altura:.2f}',
        xy=(barra.get_x() + barra.get_width() / 2, altura),
        xytext=(0, 4),
        textcoords='offset points',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.tight_layout()
# Guardado estrictamente antes de mostrar
plt.savefig('promedio_materias.png', dpi=300)
plt.show()

In [ ]:
# --- Gráfica 2: Histograma de calificaciones con línea en 70 ---
fig2, ax2 = plt.subplots(figsize=(8, 5))

ax2.hist(
    df['calificacion'], 
    bins=12, 
    color='#4a90e2', 
    edgecolor='black', 
    alpha=0.75
)

ax2.axvline(
    70, 
    color='#d90429', 
    linestyle='--', 
    linewidth=2.5, 
    label='Mínimo Aprobatorio (70)'
)

ax2.set_title('Distribución General de Calificaciones (Todas las Unidades)', pad=15, fontweight='bold')
ax2.set_xlabel('Calificación Obtenida', labelpad=10, fontweight='bold')
ax2.set_ylabel('Frecuencia (Número de Evaluaciones)', labelpad=10, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper left', frameon=True)

plt.tight_layout()
# Guardado estrictamente antes de mostrar
plt.savefig('histograma.png', dpi=300)
plt.show()

In [ ]:
# --- Gráfica 3: Líneas de tendencia por unidad por cada materia ---
fig3, ax3 = plt.subplots(figsize=(9, 5.5))
tendencia = df.groupby(['unidad', 'materia'])['calificacion'].mean().unstack()

estilos = {
    'Programacion': {'color': '#1f77b4', 'marker': 'o'},
    'Bases de datos': {'color': '#ff7f0e', 'marker': 's'},
    'Redes': {'color': '#2ca02c', 'marker': '^'},
    'Matematicas discretas': {'color': '#d62728', 'marker': 'D'}
}

for col in tendencia.columns:
    estilo = estilos.get(col, {'color': 'black', 'marker': 'o'})
    ax3.plot(
        tendencia.index, 
        tendencia[col], 
        label=col, 
        color=estilo['color'], 
        marker=estilo['marker'], 
        linewidth=2.2, 
        markersize=7
    )

ax3.set_title('Evolución y Tendencia de Calificaciones por Unidad y Materia', pad=15, fontweight='bold')
ax3.set_xlabel('Unidad de Evaluación', labelpad=10, fontweight='bold')
ax3.set_ylabel('Promedio de Calificación', labelpad=10, fontweight='bold')
ax3.set_xticks([1, 2, 3])
ax3.set_xticklabels(['Unidad 1', 'Unidad 2', 'Unidad 3'])
ax3.grid(True, linestyle='--', alpha=0.6)
ax3.legend(title='Materias Evaluadas', loc='best', frameon=True)

plt.tight_layout()
# Guardado estrictamente antes de mostrar
plt.savefig('tendencia_unidades.png', dpi=300)
plt.show()

## Ejercicio 6. Lectura de resultados y Análisis Argumentado
Con base en los resúmenes estadísticos del Ejercicio 4 y en los patrones de las visualizaciones del Ejercicio 5, se presentan las respuestas consolidadas:

1. **¿Qué materia quedó con el promedio más bajo?**  
   La materia con el promedio más bajo es **Matemáticas discretas**, con un promedio general de **65.01 puntos**.

2. **¿Si el promedio general de la unidad 3 es menor que el de las unidades 1 y 2?**  
   **Sí**, efectivamente. El promedio de la **Unidad 1 fue 75.30 puntos**, el de la **Unidad 2 fue 73.56 puntos**, mientras que en la **Unidad 3 descendió a 67.83 puntos**, situándose por debajo de ambas.

3. **¿Cuántos alumnos quedan bajo 70 en su promedio general?**  
   Un total de **6 alumnos (25.0% de los 24 alumnos evaluados)** terminaron con un promedio general inferior a 70 puntos.

---
### Párrafo de Interpretación y Justificación Cohesiva:
> *A partir del análisis cuantitativo y visual de los registros académicos, se observa que la asignatura con el rendimiento más bajo corresponde a **Matemáticas discretas**, registrando un promedio de **65.01 puntos**, situándose como la disciplina con mayor índice de dificultad conceptual para la generación. Con respecto a la progresión del curso, el promedio general experimentó un claro descenso hacia el cierre del periodo: mientras que las unidades 1 y 2 alcanzaron promedios de **75.30** y **73.56 puntos** respectivamente, la unidad 3 descendió a **67.83 puntos**. Esta caída sistemática se corrobora de manera directa en el gráfico de líneas `tendencia_unidades.png`, donde las trayectorias de las cuatro asignaturas muestran una pendiente negativa marcada hacia la tercera unidad debido al incremento acumulativo en la complejidad de los temas finales y la saturación por entregas de proyectos. Finalmente, al consolidar las calificaciones por estudiante, un total de **6 de los 24 alumnos (25.0%)** concluyeron con un promedio general inferior al mínimo aprobatorio de 70, indicando una necesidad prioritaria de implementar asesorías académicas preventivas e intervenciones remediales previo a las evaluaciones finales.*